# Module B · N3 — Calibration, the pre-declared decision, and the evidence layerThe twelve calibration lots share no lot and no component with train, so this isa genuinely held-out score and the last honest estimate the team gets before theholdout.**What this stage is not for:** redesigning the model around what it shows. TheV1 protocol allows exactly one pre-declared decision here, and it is executed insection 3.

In [ ]:
import sys, pathlibROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()sys.path.insert(0, str(ROOT))import numpy as np, pandas as pdpd.set_option("display.width", 200); pd.set_option("display.max_columns", 80)from moduleb import (baselines, config, contract, cv, dataio, envelope,                     freeze, guards, metrics, models, reason_codes)from moduleb.constants import PARAMS, TARGET_COLSfrom moduleb.features import add_features, make_foldsprint("moduleb ready — frozen config digest", config.frozen_config_digest()[:16])

## 1. Score the frozen configuration on the calibration lots

In [ ]:
tr = dataio.load_split("train"); ca = dataio.load_split("calibration")base, limits = dataio.load_specs()guards.assert_lots_disjoint(tr.frame, ca.frame, name_a="train", name_b="calibration")ftr, fca = add_features(tr.frame, base), add_features(ca.frame, base)rows, preds = [], {}for p in PARAMS:    preds[p] = models.predict_one(models.fit_one(ftr, p, config.RECOMMENDED[p]), fca, p)    mr = baselines.median_ratio_predict(baselines.median_ratio_fit(ftr, p), fca, p)    y, lots = fca[f"{p}_168h"].to_numpy(float), fca.lot_id.to_numpy()    rows.append(dict(param=p, model="MODULE_B", **metrics.metrics(y, preds[p], lots)))    rows.append(dict(param=p, model="MedianRatio_24h", **metrics.metrics(y, mr, lots)))display(pd.DataFrame(rows).round(5))

## 2. Error concentration — is a parameter's MAE really one component?A parameter where one component of 906 carries a large share of the total errordoes not have an accuracy problem spread across the fleet. It has one forecastthat went wrong, which is a different thing to fix and a different thing toreport.

In [ ]:
rows = []for p in PARAMS:    err = np.abs(preds[p] - fca[f"{p}_168h"].to_numpy(float))    o = np.argsort(err)[::-1]    rows.append(dict(param=p, MAE=err.mean(),                     worst_row_share_pct=100*err[o[0]]/err.sum(),                     top5_share_pct=100*err[o[:5]].sum()/err.sum()))display(pd.DataFrame(rows).round(3))

## 3. The pre-declared Output_Fall_Time ruleDeclared 14 Sep 2026, before `ModuleB_Calibration.csv` existed:> Move `Output_Fall_Time` to the `own` feature set **iff** `own` beats> `own+lot+cross` by more than 3% MAE **and** wins on more than half the> calibration lots.`scripts/06_predeclared_falltime_rule.py` executes it and writes the decision.Running it twice does not give two chances: the rule is spent once decided.

In [ ]:
p = dataio.RESULTS / "06_predeclared_falltime_decision.csv"display(pd.read_csv(p).T if p.exists()        else "run: python scripts/06_predeclared_falltime_rule.py")

## 4. The envelope — what may and may not be claimedMarginal coverage should land near τ. **Tail** coverage — the worst-driftingdecile — is a different quantity and is far below it.A part inside its p95 envelope is **not** thereby safe. The envelope is evidencefor fusion; the outlier judgement belongs to Module A.

In [ ]:
p = dataio.RESULTS / "09_envelope_coverage.csv"if p.exists():    cov = pd.read_csv(p)    op = cov[cov.tau == config.ENVELOPE_TAU]    display(op[["param","marginal_coverage","tail_coverage","mean_rel_width"]].round(4))else:    print("run: python scripts/09_envelope_coverage.py")

## 5. The reason codes, and the false-positive disciplineThere is no label saying "this component deserved a flag", and by design therenever will be. So the only discipline available is to measure how often each codefires and refuse to ship one that fires so often it carries no information.A flag on 2% of a lot is a worklist. A flag on 40% is wallpaper.

In [ ]:
envs = {p: envelope.fit_envelope(ftr, p, config.RECOMMENDED[p]) for p in PARAMS}env = {p: np.maximum(envelope.predict_envelope(envs[p], fca, p), preds[p]) for p in PARAMS}primary, codes, fired = reason_codes.build_reason_codes(fca, preds, limits, env)rates = reason_codes.firing_rates(fired)display(rates[rates.param == "ANY"].set_index("code")[["n_fired","rate"]].round(4))print("acceptance limits:", config.MAX_FLAG_FIRING_RATE, "per code,",      config.MAX_ANY_FLAG_RATE, "for any code")

In [ ]:
out = contract.build_output(fca, preds, limits, env)display(out.loc[out.module_b_reason_codes != "",                ["component_id","module_b_primary_parameter","module_b_reason_codes"]].head(12))

Next: **N4_freeze_and_holdout.ipynb**.